# House Prices - Feature Engineering

This notebook turns the raw Kaggle *House Prices* data into a clean, fully numeric
feature matrix ready for modelling. It continues directly from the findings of
[`00-EDA.ipynb`](00-EDA.ipynb).

**Pipeline**
1. Load raw train / test and stack them for consistent processing
2. Handle missing values (domain-aware, not blind imputation)
3. Encode ordinal quality features on a meaningful scale
4. Engineer new features (ages, total area, total baths, presence flags)
5. Reduce skewness of numeric features with `log1p`
6. One-hot encode the remaining nominal categoricals
7. Save the processed matrices to `data/processed/`

## 1. Import libraries & load data

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.stats import skew

pd.set_option("display.max_columns", 60)

DATA = Path("../data")
train_df = pd.read_csv(DATA / "train.csv")
test_df = pd.read_csv(DATA / "test.csv")
print("train:", train_df.shape, "| test:", test_df.shape)

train: (1460, 81) | test: (1459, 80)


In [2]:
# Keep ids and the (log) target aside, then stack train + test so every
# transformation is applied identically to both sets.
train_id = train_df["Id"]
test_id = test_df["Id"]
y = np.log1p(train_df["SalePrice"])          # log1p target (see EDA section 4)
n_train = len(train_df)

all_df = pd.concat(
    [train_df.drop(columns=["SalePrice"]), test_df],
    axis=0,
    ignore_index=True,
).drop(columns=["Id"])
print("combined:", all_df.shape)

combined: (2919, 79)


## 2. Missing value handling

In this dataset most "missing" values are **not** truly missing — for many features
`NaN` means *the feature is absent* (no basement, no garage, no pool …). They are therefore
imputed with domain knowledge rather than a single global rule.

In [3]:
def fill_missing(df):
    df = df.copy()

    # NaN means "feature not present" -> explicit "None" category
    none_cols = [
        "PoolQC", "MiscFeature", "Alley", "Fence", "FireplaceQu", "MasVnrType",
        "GarageType", "GarageFinish", "GarageQual", "GarageCond",
        "BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1", "BsmtFinType2",
    ]
    for c in none_cols:
        if c in df:
            df[c] = df[c].fillna("None")

    # NaN means "0 of something" -> numeric zero
    zero_cols = [
        "GarageYrBlt", "GarageArea", "GarageCars", "MasVnrArea",
        "BsmtFinSF1", "BsmtFinSF2", "BsmtUnfSF", "TotalBsmtSF",
        "BsmtFullBath", "BsmtHalfBath",
    ]
    for c in zero_cols:
        if c in df:
            df[c] = df[c].fillna(0)

    # Lot frontage is best estimated from the neighbourhood (EDA section 3)
    df["LotFrontage"] = df.groupby("Neighborhood")["LotFrontage"].transform(
        lambda x: x.fillna(x.median())
    )

    # Anything still missing: mode for categoricals, median for numerics
    for c in df.columns:
        if df[c].isna().any():
            if df[c].dtype == "object":
                df[c] = df[c].fillna(df[c].mode()[0])
            else:
                df[c] = df[c].fillna(df[c].median())
    return df


all_df = fill_missing(all_df)
print("remaining missing values:", int(all_df.isna().sum().sum()))

remaining missing values: 0


## 3. Ordinal encoding

Several categorical features are really *ordered* quality ratings. They are mapped to an
integer scale (reusing the maps from the EDA) so the model can use their ordering.

In [4]:
map_qual = {"None": 0, "Po": 1, "Fa": 2, "TA": 3, "Gd": 4, "Ex": 5}
map_expo = {"None": 0, "No": 1, "Mn": 2, "Av": 3, "Gd": 4}
map_fin  = {"None": 0, "Unf": 1, "LwQ": 2, "Rec": 3, "BLQ": 4, "ALQ": 5, "GLQ": 6}
map_func = {"Sev": 1, "Maj2": 2, "Maj1": 3, "Mod": 4, "Min2": 5, "Min1": 6, "Typ": 7}

col2map = {
    "ExterQual": map_qual, "ExterCond": map_qual,
    "BsmtQual": map_qual, "BsmtCond": map_qual, "BsmtExposure": map_expo,
    "BsmtFinType1": map_fin, "BsmtFinType2": map_fin,
    "HeatingQC": map_qual, "KitchenQual": map_qual, "FireplaceQu": map_qual,
    "GarageQual": map_qual, "GarageCond": map_qual, "PoolQC": map_qual,
    "Functional": map_func,
}

for col, mapping in col2map.items():
    if col in all_df:
        all_df[col] = all_df[col].map(mapping).fillna(0).astype(int)

print("ordinal features encoded:", len(col2map))

ordinal features encoded: 14


## 4. Feature engineering

A handful of features that encode domain knowledge and tend to be strong
predictors of price are added: property ages, total living area, total bathroom count,
total porch area, and binary "has-X" flags.

In [5]:
# MSSubClass is a code, not a quantity -> treat as categorical
all_df["MSSubClass"] = all_df["MSSubClass"].astype(str)

# Ages at time of sale
all_df["HouseAge"] = all_df["YrSold"] - all_df["YearBuilt"]
all_df["RemodAge"] = all_df["YrSold"] - all_df["YearRemodAdd"]
all_df["GarageAge"] = all_df["YrSold"] - all_df["GarageYrBlt"]
all_df.loc[all_df["GarageYrBlt"] == 0, "GarageAge"] = 0   # no garage -> age 0
for c in ["HouseAge", "RemodAge", "GarageAge"]:
    all_df[c] = all_df[c].clip(lower=0)                    # guard tiny negatives

# Aggregated size / count features
all_df["TotalSF"] = all_df["TotalBsmtSF"] + all_df["1stFlrSF"] + all_df["2ndFlrSF"]
all_df["TotalBath"] = (
    all_df["FullBath"] + 0.5 * all_df["HalfBath"]
    + all_df["BsmtFullBath"] + 0.5 * all_df["BsmtHalfBath"]
)
all_df["TotalPorchSF"] = all_df[
    ["OpenPorchSF", "EnclosedPorch", "3SsnPorch", "ScreenPorch", "WoodDeckSF"]
].sum(axis=1)

# Presence flags
all_df["HasPool"] = (all_df["PoolArea"] > 0).astype(int)
all_df["Has2ndFloor"] = (all_df["2ndFlrSF"] > 0).astype(int)
all_df["HasGarage"] = (all_df["GarageArea"] > 0).astype(int)
all_df["HasBsmt"] = (all_df["TotalBsmtSF"] > 0).astype(int)
all_df["HasFireplace"] = (all_df["Fireplaces"] > 0).astype(int)

print("shape after feature engineering:", all_df.shape)
all_df[["TotalSF", "TotalBath", "TotalPorchSF", "HouseAge", "RemodAge"]].head()

shape after feature engineering: (2919, 90)


,TotalSF,TotalBath,TotalPorchSF,HouseAge,RemodAge
0,2566.0,3.5,61,5,5
1,2524.0,2.5,298,31,31
2,2706.0,3.5,42,7,6
3,2473.0,2.0,307,91,36
4,3343.0,3.5,276,8,8


## 5. Reduce skewness

Linear models assume roughly symmetric inputs. Heavily right-skewed numeric features
are compressed with `log1p` (the same idea applied to the target).

In [6]:
numeric_feats = all_df.select_dtypes(include=[np.number]).columns
skewness = all_df[numeric_feats].apply(lambda x: skew(x.dropna())).sort_values(ascending=False)
high_skew = [c for c in skewness[abs(skewness) > 0.75].index if all_df[c].min() >= 0]

for c in high_skew:
    all_df[c] = np.log1p(all_df[c])

print(f"{len(high_skew)} skewed features log-transformed")
skewness.head(8)

37 skewed features log-transformed


MiscVal         21.947195
PoolQC          18.409137
PoolArea        16.898328
HasPool         14.884318
LotArea         12.822431
LowQualFinSF    12.088761
3SsnPorch       11.376065
KitchenAbvGr     4.302254
dtype: float64

## 6. One-hot encode nominal categoricals

In [7]:
all_df = pd.get_dummies(all_df, drop_first=True, dtype=int)
print("final feature matrix:", all_df.shape)

final feature matrix: (2919, 236)


## 7. Split back & save processed data

In [8]:
X_train = all_df.iloc[:n_train].reset_index(drop=True)
X_test = all_df.iloc[n_train:].reset_index(drop=True)

PROC = DATA / "processed"
PROC.mkdir(exist_ok=True)

X_train.assign(SalePrice_Log1p=y.values).to_csv(PROC / "train_processed.csv", index=False)
X_test.to_csv(PROC / "test_processed.csv", index=False)
pd.DataFrame({"Id": test_id}).to_csv(PROC / "test_ids.csv", index=False)

print("saved -> data/processed/")
print("X_train:", X_train.shape, "| X_test:", X_test.shape)

saved -> data/processed/
X_train: (1460, 236) | X_test: (1459, 236)


### Output

`data/processed/` now contains a fully numeric, missing-free feature matrix for both
train and test, plus the test ids. Continue to
[`02-Modeling.ipynb`](02-Modeling.ipynb).